# Textual Entailment Fact Checking

Use a pre-trained DeBERTa-v3 NLI model (MNLI + FEVER + ANLI) to decide whether each fact (hypothesis) is entailed by a passage sentence (premise). The three-way output (entailment / neutral / contradiction) is mapped to S / NS. Target should be more than or equal to 83% accuracy.

**Plan is to** run the model once over every (sentence, fact) pair -> cache the probabilities -> tune decision rules, thresholds and pruning offline.

## 1. Imports and paths

In [2]:
import sys
import re
import json
import time
import gc
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Project root (two levels up) so we can import factcheck.py and factchecking_main.py directly.
ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

# Reuse the official loaders
from factchecking_main import read_passages, read_fact_examples
# Reuse the Part 1 word-overlap scorer for pruning.
from factcheck import WordRecallThresholdFactChecker

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

/Users/seyedrumaiz/Library/Mobile Documents/com~apple~CloudDocs/University/Year 3/CM3604/LLMFactChecking/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load data (loaders of factchecking_main.py)

In [4]:
# Load the data and convert to binary labels (S vs NS, where NS includes IR)
fact_to_passage = read_passages(str(ROOT / "data/passages_bm25_ChatGPT_humfacts.jsonl"))
examples = read_fact_examples(str(ROOT / "data/dev_labeled_ChatGPT.jsonl"), fact_to_passage)

# IR -> NS, exactly like predict_two_classes() in factchecking_main.py
y_true = np.array([1 if ex.label == "S" else 0 for ex in examples])

print(f"Examples: {len(examples)}  |  S = {y_true.sum()}  |  NS (incl. IR) = {(y_true == 0).sum()}")

Examples: 221  |  S = 112  |  NS (incl. IR) = 109


## 3. Load the entailment model

In [5]:
MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# Confirm the label order instead of assuming it.
print(model.config.id2label)

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## 4. Smoke test on one pair

In [6]:
def entailment_probs(premises, hypotheses, batch_size=16):

    all_probs = []
    with torch.no_grad():
        for start in range(0, len(premises), batch_size):
            inputs = tokenizer(
                premises[start:start + batch_size],
                hypotheses[start:start + batch_size],
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            logits = model(**inputs).logits
            all_probs.append(torch.softmax(logits, dim=-1).numpy())
            del inputs, logits
    gc.collect()
    return np.vstack(all_probs)

In [7]:
tests = [
    ("Lanny Flaherty (born July 27, 1942) is an American actor.", "Lanny Flaherty is an American."),       # entail
    ("Lanny Flaherty (born July 27, 1942) is an American actor.", "Lanny Flaherty was born in 1950."),     # contradict
    ("Lanny Flaherty (born July 27, 1942) is an American actor.", "Lanny Flaherty plays the guitar."),     # be neutral
]

In [8]:
probs = entailment_probs([p for p, _ in tests], [h for _, h in tests])
for (p, h), pr in zip(tests, probs):
    print(f"E={pr[0]:.3f}  N={pr[1]:.3f}  C={pr[2]:.3f}  |  {h}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


E=0.998  N=0.002  C=0.000  |  Lanny Flaherty is an American.
E=0.000  N=0.000  C=0.999  |  Lanny Flaherty was born in 1950.
E=0.000  N=0.997  C=0.003  |  Lanny Flaherty plays the guitar.


## 5. Sentence splitting and cleaning

In [9]:
def split_passage(text):
    sentences = []
    for chunk in re.split(r"</?s>", text):
        chunk = chunk.strip()
        if not chunk:
            continue
        for sent in re.split(r"(?<=[.!?])\s+", chunk):
            sent = re.sub(r"\s+", " ", sent).strip()
            if len(sent.split()) >= 3:  # drop noise like "Career." or ""
                sentences.append(sent)
    return sentences

In [10]:
ex = examples[0]
print("FACT:", ex.fact)

FACT: Marianne McAndrew is an American.


In [11]:
for s in split_passage(ex.passages[0]["text"])[:5]:
    print("  -", s)

  - Marianne McAndrew Marianne Christine McAndrew is an actress known for her role as Irene Molloy in the film "Hello, Dolly!"
  - "Hello, Dolly!" was McAndrew's first credited film role.
  - The role of Irene Molloy was given considerably more attention in the film than in earlier Broadway productions.
  - "Hello, Dolly!" earned McAndrew two Golden Globe nominations in 1969; Golden Globe Award for Best Supporting Actress – Motion Picture and the since discontinued Golden Globe Award for New Star of the Year – Actress, as well as generally good reviews.
  - She landed a starring (second billed) role in her next film ("The Seven Minutes").


## 6. Run the model once on every pair and cache the results

In [12]:
# Part 1 IDF-recall, reused for pruning later
overlap_scorer = WordRecallThresholdFactChecker()

In [13]:
rows = []

In [14]:
for i, ex in enumerate(examples):
    for p_idx, passage in enumerate(ex.passages):
        sents = split_passage(passage["text"])
        for s_idx, sent in enumerate(sents):
            rows.append({
                "fact_idx": i,
                "passage_idx": p_idx,
                "sent_idx": s_idx,
                "premise": sent,
                # 2-sentence window (previous + current) to help with pronouns like "He was born..."
                "premise_window": (sents[s_idx - 1] + " " + sent) if s_idx > 0 else sent,
                "overlap": overlap_scorer.score(ex.fact, [{"text": sent}]),
            })

In [15]:
print(f"Total pairs: {len(rows)}")

Total pairs: 5417


In [16]:
hypotheses = [examples[r["fact_idx"]].fact for r in rows]

In [17]:
start = time.time()
probs_sent = entailment_probs([r["premise"] for r in rows], hypotheses)
print(f"Sentence premises done in {time.time() - start:.0f}s")

Sentence premises done in 279s


In [18]:
start = time.time()
probs_window = entailment_probs([r["premise_window"] for r in rows], hypotheses)
print(f"Window premises done in {time.time() - start:.0f}s")

Window premises done in 328s


In [19]:
# Save everything so later experiments never need to re-run the model.
np.savez(
    CACHE_DIR / "pair_probs.npz",
    fact_idx=np.array([r["fact_idx"] for r in rows]),
    overlap=np.array([r["overlap"] for r in rows]),
    probs_sent=probs_sent,
    probs_window=probs_window,
    y_true=y_true,
)
with open(CACHE_DIR / "pairs.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print("Cached to", CACHE_DIR)

Cached to cache


## 7. Load cached results and helpers

In [17]:
# Load the cached model outputs
cache = np.load(CACHE_DIR / "pair_probs.npz")
fact_idx = cache["fact_idx"]       # which fact each pair belongs to
pair_overlap = cache["overlap"]      # per-sentence word overlap (Part 1 scorer)
probs_sent = cache["probs_sent"]      # [P(E), P(N), P(C)] with single-sentence premise
probs_window = cache["probs_window"]  # same, with previous + current sentence as premise
N_FACTS = len(y_true)

In [18]:
def max_per_fact(values, mask=None, default=0.0):
    """Fact score = max over its (kept) sentence pairs, as the brief requires."""
    keep = np.ones(len(values), dtype=bool) if mask is None else mask
    scores = np.full(N_FACTS, default, dtype=float)
    np.maximum.at(scores, fact_idx[keep], values[keep])
    return scores

In [19]:
def accuracy(pred):
    return float((pred == y_true).mean())

In [20]:
def tune_threshold(scores, lo=0.0, hi=1.0, steps=1001):
    grid = np.linspace(lo, hi, steps)
    accs = np.array([accuracy((scores >= t).astype(int)) for t in grid])
    best = int(np.argmax(accs))
    return float(grid[best]), float(accs[best])

In [21]:
def confusion(pred):
    return {
        "TP": int(((pred == 1) & (y_true == 1)).sum()),
        "FN": int(((pred == 0) & (y_true == 1)).sum()),
        "FP": int(((pred == 1) & (y_true == 0)).sum()),
        "TN": int(((pred == 0) & (y_true == 0)).sum()),
    }